# Exercise 2: PyTorch core

In this exercise you’ll build core PyTorch “muscle memory” that you’ll reuse in basically every model you write:

- **Autograd**: how gradients are created, how they accumulate, and how to compute gradients for one or multiple inputs.
- **Dataloading**: writing small `Dataset`s, using `DataLoader`, and custom `collate_fn`.
- **Optimizers**: implementing **AdamW** updates from scratch (state, bias correction, weight decay).
- **Training basics**: a clean single training step.
- **Initialization**: fan-in/out and common initializers (Xavier / Kaiming), plus a helper to init `nn.Linear`.

As before: fill in all `TODO`s without changing function names or signatures.
When debugging, print shapes/dtypes/devices, and write tiny sanity checks (e.g. compare to PyTorch’s built-ins).


In [2]:
from __future__ import annotations
from dataclasses import dataclass
import torch
from torch import nn

## Autograd fundamentals

PyTorch builds a computation graph when you apply operations to tensors with `requires_grad=True`.
Calling `backward()` (or `torch.autograd.grad`) computes gradients by traversing that graph.

### Key concepts
- **Leaf tensor**: a tensor created by you (not the result of an operation) with `requires_grad=True`. Leaf tensors can store gradients in `.grad`.
- **Gradient accumulation**: calling `backward()` adds into `.grad` (it does not overwrite). You must reset gradients between steps/calls.
- **`torch.autograd.grad` vs `.backward()`**
  - `torch.autograd.grad(f, x)` returns `df/dx` directly and does not write into `x.grad` unless you explicitly do so.
  - `f.backward()` writes gradients into `.grad` of leaf tensors.

In the next functions you’ll compute gradients for a simple scalar function such as `f(x) = sum(x^2)` using both APIs.

### `torch.no_grad()`
Wrap inference-only code to avoid tracking gradients and building graphs:
- saves memory
- speeds up evaluation

### `detach()`
`y = x.detach()` returns a tensor that shares data with `x` but is **not connected** to the autograd graph.
This is useful when you want to treat something as a constant target.

### `model.train()` vs `model.eval()`
- `train()` enables training behavior (e.g. dropout active, batchnorm updates running stats).
- `eval()` enables inference behavior (e.g. dropout off, batchnorm uses running stats).

In [3]:
def grad_with_autograd_grad(x: torch.Tensor) -> torch.Tensor:
    """
    Compute gradient of f(x) = sum(x^2) using torch.autograd.grad

    Requirements:
    - Do not call .backward().
    - x should require grad inside the function (don't assume it does).
    - Must return df/dx
    """
    # TODO: implement
    x.requires_grad = True
    f = sum(x ** 2)
    return torch.autograd.grad(f,x)

x = torch.tensor([1,2,3], dtype=torch.float32)
print(grad_with_autograd_grad(x))

(tensor([2., 4., 6.]),)


In [4]:

def grad_with_backward(x: torch.Tensor) -> torch.Tensor:
    """
    Compute gradient of f(x) = sum(x^2) using .backward().

    Requirements:
    - Must return df/dx
    - Must not leak gradients across calls (watch x.grad accumulation)
    """
    # TODO: implement
    x.grad = None # Reset gradients to prevent across call accumulation
    f = sum(x ** 2)
    return f.backward()

x = torch.tensor([1,2,3], dtype=torch.float32)
print(grad_with_autograd_grad(x))

(tensor([2., 4., 6.]),)


In [5]:
def grad_wrt_multiple_inputs(
    a: torch.Tensor, b: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Compute gradients w.r.t. multiple inputs. The function is f(a, b) = sum(a^2 + ab).

    Return:
        (df/da, df/db)

    Requirements:
    - Use torch.autograd.grad
    - Ensure both a and b require grad in this function.
    """
    # TODO: implement
    a.requires_grad = True
    b.requires_grad = True
    f = torch.sum(a**2 + a*b)
    return torch.autograd.grad(outputs=f, inputs=[a, b])

a = torch.tensor([1,2,3], dtype=torch.float32)
b = torch.tensor([4,5,6], dtype=torch.float32)
print(grad_wrt_multiple_inputs(a,b))

(tensor([ 6.,  9., 12.]), tensor([1., 2., 3.]))


## Dataloading

In PyTorch, a `Dataset` defines how to fetch a *single* training example, and a `DataLoader` handles:
- batching
- shuffling
- parallel workers
- optional custom batching logic via `collate_fn`

### `Dataset` in one sentence
A `Dataset` only needs:
- `__len__`: number of items
- `__getitem__`: return one item (e.g. `(x, y)`)

### Why `collate_fn` matters
The default DataLoader collation stacks items along a new batch dimension.
That works for fixed-size tensors, but it breaks for **variable-length sequences**.

So we’ll implement padding ourselves:
- Convert a list of 1D token sequences into a padded tensor `(B, T_max)`
- Track `lengths` and a `padding_mask`

### Mask convention for padding
For padding masks in this exercise:
- `padding_mask[b, t] == True` means **this is padding / invalid**
- `padding_mask[b, t] == False` means **this is a real token**

In [6]:
from torch.utils.data import DataLoader, Dataset

In [7]:
class TensorPairDataset(Dataset):
    """
    Minimal dataset wrapping (x, y).

    x: (N, ...)
    y: (N, ...)

    N is the number of samples. The dataset should return tuples of (x[i], y[i]).
    """

    def __init__(self, x: torch.Tensor, y: torch.Tensor):
        # TODO: implement
        self.x = x
        self.y = y

    def __len__(self) -> int:
        # TODO: implement
        return self.x.size(0)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        # TODO: implement
        return (self.x[idx], self.y[idx])

In [8]:
class NextTokenDataset(Dataset):
    """
    Next-token prediction dataset.

    Given tokens of shape (N, T), produce:
      input_ids  = tokens[:, :-1]
      target_ids = tokens[:, 1:]

    Return per item:
      (input_ids, target_ids)

    Notes:
    - Returned tensors should be 1D of length (T-1).
    - dtype should remain integer.
    """

    def __init__(self, tokens: torch.Tensor):
        # TODO: implement
        self.tokens = tokens
        self.inputs_ids = tokens[:, :-1]
        self.target_ids = tokens[:, 1:]

        print(tokens)
        print(self.inputs_ids)
        print(self.target_ids)

    def __len__(self) -> int:
        # TODO: implement
        return self.tokens.size(0)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        # TODO: implement
        return (self.inputs_ids[idx],self.target_ids[idx])
    

N = 2
T = 4

x = torch.randint(low=0,high=10,size=(N,T))
data = NextTokenDataset(x)
print(len(data))
print(data[0])

tensor([[7, 1, 6, 3],
        [3, 1, 8, 1]])
tensor([[7, 1, 6],
        [3, 1, 8]])
tensor([[1, 6, 3],
        [1, 8, 1]])
2
(tensor([7, 1, 6]), tensor([1, 6, 3]))


In [ ]:

class RandomCropSequenceDataset(Dataset):
    """
    Sequence dataset that returns random crops of fixed length.

    tokens: (N, T_total)
    crop_len: L

    For each __getitem__:
      - sample a start index s so that s+L <= T_total
      - return tokens[idx, s:s+L]

    Requirements:
    - Use a torch.Generator for deterministic behavior if seed is provided.
    - Do NOT use Python's random module.
    """

    def __init__(self, tokens: torch.Tensor, crop_len: int, seed: int | None = None):
        # TODO: implement
        self.tokens = tokens
        self.crop_len = crop_len
        self.rand = torch.Generator()
        self.rand.manual_seed(seed)

    def __len__(self) -> int:
        # TODO: implement
        return self.tokens.size(0)

    def __getitem__(self, idx: int) -> torch.Tensor:
        # TODO: implement
        s = torch.randint(low=0, high=self.tokens.size(1) - self.crop_len+1, size=(1,), generator=self.rand).item()
        return self.tokens[idx, s:s+self.crop_len]
    
N = 2
T = 5
L = 2

x = torch.randint(low=0,high=10,size=(N,T))
data = RandomCropSequenceDataset(x,L,42)
print(len(data))
print(data[0])

2
tensor([7, 3])


In [68]:


@dataclass(frozen=True)
class PaddedBatch:
    """
    A padded batch for variable-length sequences.

    tokens: LongTensor (B, T_max)
    lengths: LongTensor (B,)
    padding_mask: BoolTensor (B, T_max) where True means "this is padding"
    """

    tokens: torch.Tensor
    lengths: torch.Tensor
    padding_mask: torch.Tensor


def pad_1d_sequences(seqs: list[torch.Tensor], pad_value: int = 0) -> PaddedBatch:
    """
    Pad a list of 1D integer tensors to the same length.

    Requirements:
    - Return PaddedBatch(tokens, lengths, padding_mask)
    - padding_mask[b, t] == True iff t >= lengths[b]
    - tokens should be dtype long, if not cast them
    """
    # TODO: implement
    lengths =  torch.tensor([t.numel() for t in seqs], dtype=torch.long)
    T_max = lengths.max().item()
    B = len(lengths)

    mask = torch.arange(T_max) >= lengths[:, None]
    tokens = torch.full(size=(B,T_max), fill_value=pad_value, dtype=torch.long)

    for b,t in enumerate(seqs):
        t_long = t.to(dtype=torch.long)
        tokens[b,:lengths[b]] = t_long
    
    return PaddedBatch(tokens, lengths, mask)
 
test_1 = [torch.tensor([1,2,3]), torch.tensor([1,2]), torch.tensor([1,4,5,6])]
test_2 = [torch.tensor([1,2,4,5]), torch.tensor([1,2]), torch.tensor([1,4,5,6,9,10])]
test_3 = [torch.tensor([1,2,3]), torch.tensor([1,2,4]), torch.tensor([1,4,5])]
print(pad_1d_sequences(test_1))
print(pad_1d_sequences(test_2))
print(pad_1d_sequences(test_3))

PaddedBatch(tokens=tensor([[1, 2, 3, 0],
        [1, 2, 0, 0],
        [1, 4, 5, 6]]), lengths=tensor([3, 2, 4]), padding_mask=tensor([[False, False, False,  True],
        [False, False,  True,  True],
        [False, False, False, False]]))
PaddedBatch(tokens=tensor([[ 1,  2,  4,  5,  0,  0],
        [ 1,  2,  0,  0,  0,  0],
        [ 1,  4,  5,  6,  9, 10]]), lengths=tensor([4, 2, 6]), padding_mask=tensor([[False, False, False, False,  True,  True],
        [False, False,  True,  True,  True,  True],
        [False, False, False, False, False, False]]))
PaddedBatch(tokens=tensor([[1, 2, 3],
        [1, 2, 4],
        [1, 4, 5]]), lengths=tensor([3, 3, 3]), padding_mask=tensor([[False, False, False],
        [False, False, False],
        [False, False, False]]))


In [75]:
def collate_next_token_batch(
    batch: list[tuple[torch.Tensor, torch.Tensor]], pad_value: int = 0
) -> dict[str, torch.Tensor]:
    """
    Collate for NextTokenDataset samples that may have variable lengths.

    batch: list of (input_ids, target_ids), each 1D

    Return dict with:
      - input_ids: (B, T_max)
      - target_ids: (B, T_max)
      - attention_mask: (B, T_max) where True means "keep" (NOT padding)
      - padding_mask: (B, T_max) where True means "padding"

    Requirements:
    - pad input_ids and target_ids consistently
    - attention_mask is the logical NOT of padding_mask
    """
    # TODO: implement
    input_seqs, target_seqs = zip(*batch)
    input_pad = pad_1d_sequences(input_seqs,pad_value)
    target_pad = pad_1d_sequences(target_seqs,pad_value)

    attention_mask = ~input_pad.padding_mask

    print(f"Input and target id's padding are consistent? : \
          {(input_pad.padding_mask == target_pad.padding_mask).all().item()}")

    return {
        "input_ids": input_pad.tokens,
        "target_ids": target_pad.tokens,
        "attention_mask": attention_mask,
        "padding_mask": input_pad.padding_mask
    }

input_1 = (torch.tensor([1,2]), torch.tensor([2,3]))
input_2 = (torch.tensor([2,3,1]), torch.tensor([3,4,5]))
input_3 = (torch.tensor([4,5,6,7,8,9,10]), torch.tensor([5,6,7,8,9,10,11]))
input_4 = (torch.tensor([4,5,6,7,9,10]), torch.tensor([5,6,8,9,10,11]))

x_1 = [input_1, input_2, input_3]
x_2 = [input_1, input_4]
x_3 = [input_4, input_2, input_1, input_3]
print(collate_next_token_batch(x_1))
print(collate_next_token_batch(x_2))
print(collate_next_token_batch(x_3))

Input and target id's padding are consistent? :           True
{'input_ids': tensor([[ 1,  2,  0,  0,  0,  0,  0],
        [ 2,  3,  1,  0,  0,  0,  0],
        [ 4,  5,  6,  7,  8,  9, 10]]), 'target_ids': tensor([[ 2,  3,  0,  0,  0,  0,  0],
        [ 3,  4,  5,  0,  0,  0,  0],
        [ 5,  6,  7,  8,  9, 10, 11]]), 'attention_mask': tensor([[ True,  True, False, False, False, False, False],
        [ True,  True,  True, False, False, False, False],
        [ True,  True,  True,  True,  True,  True,  True]]), 'padding_mask': tensor([[False, False,  True,  True,  True,  True,  True],
        [False, False, False,  True,  True,  True,  True],
        [False, False, False, False, False, False, False]])}
Input and target id's padding are consistent? :           True
{'input_ids': tensor([[ 1,  2,  0,  0,  0,  0],
        [ 4,  5,  6,  7,  9, 10]]), 'target_ids': tensor([[ 2,  3,  0,  0,  0,  0],
        [ 5,  6,  8,  9, 10, 11]]), 'attention_mask': tensor([[ True,  True, False, False,

In [ ]:
def make_dataloader(
    dataset: Dataset,
    batch_size: int,
    shuffle: bool = True,
    drop_last: bool = False,
    collate_fn=None,
    num_workers: int = 0,
) -> DataLoader:
    """
    Create a DataLoader with optional collate_fn.
    """
    # TODO: implement
    return DataLoader(dataset=dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, collate_fn=collate_fn,
                      num_workers=num_workers)

N = 8
T = 6
L = 4
batch_size = N // 2

x = torch.randint(low=0,high=10,size=(N,T))
#data = NextTokenDataset(x)

data = RandomCropSequenceDataset(x,L,42)

test_loader = make_dataloader(dataset=data, batch_size=batch_size, shuffle=True, 
                              drop_last=False, collate_fn=collate_next_token_batch, num_workers=0)

## TO BE COMPLETED!!!

## Optimizers (AdamW from scratch)

PyTorch optimizers keep **state** for each parameter (e.g. moment estimates in Adam).
In this section you’ll implement **AdamW**, which is Adam + *decoupled* weight decay.

### AdamW state
For each parameter tensor `p` we store:
- `m`: first moment (EMA of gradients)
- `v`: second moment (EMA of squared gradients)
- `t`: step counter

### Update overview (high level)
1) Update moments `m, v`
2) Bias-correct them (`m_hat, v_hat`)
3) Apply parameter update:
   `p -= lr * ( m_hat / (sqrt(v_hat) + eps) + weight_decay * p )`

Notes:
- This update is **in-place** (mutates `p`).
- Gradients should not be modified.
- State tensors must match parameter shape/device/dtype.

In [ ]:
@dataclass
class AdamWState:
    """
    Per-parameter AdamW state.

    m: first moment
    v: second moment
    t: step count
    """

    m: torch.Tensor
    v: torch.Tensor
    t: int


def init_adamw_state(p: torch.Tensor) -> AdamWState:
    """
    Initialize AdamW state tensors for a parameter tensor p.

    What to create:
    - m: zeros like p, same shape/device/dtype
    - v: zeros like p, same shape/device/dtype
    - t: step counter starting at 0

    Notes / requirements:
    - Use torch.zeros_like(p) for m and v.
    - Do NOT attach gradients to the state (initialize under torch.no_grad()).
    - t starts at 0. In adamw_step_, increment t to 1 on the first update *before*
      computing bias correction terms (1 - beta1^t) and (1 - beta2^t).
    - State tensors must live on the same device as p (CPU vs GPU) and have the
      same dtype as p.
    """
    # TODO: implement
    with torch.no_grad():
      m = torch.zeros_like(p) # *_like builders also copy the dtype and device
      v = torch.zeros_like(v)
      t = 0
    return AdamWState(m,v,t)
      

In [92]:
def adamw_step_(
    p: torch.Tensor,
    grad: torch.Tensor,
    state: AdamWState,
    lr: float,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-8,
    weight_decay: float = 0.01,
) -> AdamWState:
    """
    In-place AdamW parameter update (updates p).

    Algorithm (AdamW):
      m = beta1*m + (1-beta1)*grad
      v = beta2*v + (1-beta2)*grad^2
      m_hat = m / (1 - beta1^t)
      v_hat = v / (1 - beta2^t)
      p = p - lr * (m_hat / (sqrt(v_hat) + eps) + weight_decay * p)

    Requirements:
    - Update p in-place.
    - Return updated state (with incremented t).
    - Do not modify grad.
    - Should work for any tensor shape.
    """
    # TODO: implement
    state.t += 1
    state.m = betas[0] * state.m + (1 - betas[0])*grad
    state.v = betas[1] * state.v + (1 - betas[1])*(grad ** 2)
    m_hat = state.m / (1 - betas[0]**state.t)
    v_hat = state.v / (1 - betas[1]**state.t)
    p.mul_(1 - lr * weight_decay)
    p.sub_(other= m_hat / (torch.sqrt(v_hat) + eps), alpha=lr)
    return state
    

In [93]:
def adamw_step_many_(
    params: list[torch.Tensor],
    grads: list[torch.Tensor],
    states: list[AdamWState],
    lr: float,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-8,
    weight_decay: float = 0.01,
) -> list[AdamWState]:
    """
    Apply AdamW to many parameters.

    Requirements:
    - len(params) == len(grads) == len(states)
    - Update each param in-place.
    - Return the list of updated states.
    """
    # TODO: implement
    N = len(params)
    for i in range(N):
        states[i].t += 1
        states[i].m = betas[0] * states[i].m + (1 - betas[0])*grads[i]
        states[i].v = betas[1] * states[i].v + (1 - betas[1])*(grads[i] ** 2)
        m_hat = states[i].m / (1 - betas[0]**states[i].t)
        v_hat = states[i].v / (1 - betas[1]**states[i].t)
        params[i].mul_(1 - lr * weight_decay)
        params[i].sub_(other=m_hat / (torch.sqrt(v_hat) + eps), alpha=lr)
    return states



## Training basics

A minimal training step follows the same pattern almost everywhere:

1) set model to train mode
2) reset gradients
3) forward pass
4) compute loss
5) backward pass
6) step optimizer

In this exercise you’ll implement a single MSE training step using a standard PyTorch optimizer.
Return a Python float loss value.

In [94]:
def train_step_mse(
    model: nn.Module,
    batch: tuple[torch.Tensor, torch.Tensor],
    optimizer: torch.optim.Optimizer,
) -> float:
    """
    One MSE train step using standard torch optimizer.
    """
    # TODO: implement
    model.train()
    x, y = batch
    pred = model(x)
    loss = torch.nn.functional.mse_loss(pred,y).float()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss


## Parameter initialization

Initialization matters because it controls signal and gradient scales at the start of training.

### Fan-in / fan-out
- `fan_in`: number of input connections to a unit
- `fan_out`: number of output connections from a unit

For a Linear layer weight of shape `(out_features, in_features)`:
- `fan_in = in_features`
- `fan_out = out_features`

### Common schemes
- **Xavier / Glorot** (often good for tanh / linear-ish nets):
  keeps variance stable across layers when activations are roughly symmetric.
- **Kaiming / He** (often good for ReLU-like nets):
  accounts for the fact that ReLU zeroes out about half the inputs.

In this section you’ll implement Xavier uniform and Kaiming uniform and use them to initialize `nn.Linear`.
We also always zero the bias unless explicitly told otherwise.

In [ ]:
def fan_in_fan_out(weight: torch.Tensor) -> tuple[int, int]:
    """Compute (fan_in, fan_out) for a weight tensor."""
    # TODO: implement
    return (weight.size(1), weight.size(0))

w = torch.tensor([[1,2], [5,3], [6,1]])
print(fan_in_fan_out(w))

torch.Size([3, 2])
(2, 3)


In [112]:
def xavier_uniform_(weight: torch.Tensor, gain: float = 1.0) -> torch.Tensor:
    """
    In-place Xavier/Glorot uniform init:
      bound = gain * sqrt(6 / (fan_in + fan_out))
      U(-bound, bound)
    """
    # TODO: implement
    fan_in, fan_out = fan_in_fan_out(weight)
    bound = (gain * (6 / (fan_in + fan_out)) ** (0.5))
    print(f"My bound is: {bound}")
    weight.uniform_(-bound, bound)
    return w

w = torch.tensor([[1,3], [5,10], [7,1]], dtype=torch.float32)
print(xavier_uniform_(w))

My bound is: 1.0954451150103321
tensor([[-0.2046, -0.4639],
        [ 0.0192,  0.3816],
        [ 0.1910,  0.8871]])


In [115]:
def kaiming_uniform_(weight: torch.Tensor, nonlinearity: str = "relu") -> torch.Tensor:
    """
    In-place Kaiming/He uniform init.

    Follow this common choice:
      gain = sqrt(2) for ReLU
      std = gain / sqrt(fan_in)
      bound = sqrt(3) * std
      U(-bound, bound)
    """
    # TODO: implement
    fan_in, fan_out = fan_in_fan_out(weight)
    if nonlinearity == "relu":
        gain = 2.0 ** 0.5
    elif nonlinearity == "tanh":
        gain = 5.0 / 3
    else:
        gain = 1
    std = gain / (fan_in ** 0.5)
    bound = (3 ** 0.5) * std
    print(f"The bound is: {bound}")
    weight.uniform_(-bound,bound)
    return weight

w = torch.tensor([[1,3], [5,10], [7,1]], dtype=torch.float32)
print(kaiming_uniform_(w))


The bound is: 1.7320508075688772
tensor([[-1.2912,  0.6940],
        [ 1.0389, -0.2688],
        [ 1.2185, -0.6448]])


In [124]:
def init_linear_(layer: nn.Linear, scheme: str = "xavier") -> nn.Linear:
    """
    Initialize an nn.Linear in-place.

    scheme:
      - "xavier"
      - "kaiming_relu"
      - "zero" (weights and bias = 0)
    """
    # TODO: implement
    with torch.no_grad():
        if scheme == "xavier":
            xavier_uniform_(layer.weight)
        elif scheme == "kaiming_relu":
            kaiming_uniform_(layer.weight)
        else:
            layer.weight.fill_(0.0)
            layer.bias.fill_(0.0)
  
        return layer

layer = nn.Linear(in_features=3, out_features=6, dtype=torch.float32)

xavier = init_linear_(layer, "xavier")
he = init_linear_(layer,"kaiming_relu")
zero = init_linear_(layer, "zero")

print(f"Xavier: Weights: {xavier.weight} Bias: {xavier.bias} \n\n He: Weights: {he.weight} Bias: {he.bias} \n\n \
      Zero: Weights: {zero.weight} Bias: {zero.bias}")

My bound is: 0.816496580927726
The bound is: 1.4142135623730951
Xavier: Weights: Parameter containing:
tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]], requires_grad=True) Bias: Parameter containing:
tensor([0., 0., 0., 0., 0., 0.], requires_grad=True) 

 He: Weights: Parameter containing:
tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]], requires_grad=True) Bias: Parameter containing:
tensor([0., 0., 0., 0., 0., 0.], requires_grad=True) 

       Zero: Weights: Parameter containing:
tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]], requires_grad=True) Bias: Parameter containing:
tensor([0., 0., 0., 0., 0., 0.], requires_grad=True)
